# Python for AI — Class 11
### Inheritance and polymorphism — OOP part 2

Class 10 gave you one class at a time. Real programs have *families* of classes: a savings account and a current account are both bank accounts, a dog and a cat are both animals. **Inheritance** lets a class reuse another's code instead of copying it, and **polymorphism** lets one line of code work across the whole family. Then **dunder methods** make your objects behave like Python's own — printing nicely, comparing with `==`.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

# 1. Inheritance — a child class reuses a parent

Write the shared behaviour once in a **parent** (or **base**) class. A **child** class written as `class Child(Parent):` gets every one of the parent's methods for free, and can add its own.

Here `Account` holds what every account has. `SavingsAccount` *is an* `Account` — it inherits `deposit` and `withdraw` without repeating them, and adds interest on top.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            print(f"Refused: {self.owner} has only Rs {self.balance}")
            return
        self.balance -= amount


class SavingsAccount(Account):        # SavingsAccount IS AN Account
    def add_interest(self, rate):
        self.balance += self.balance * rate

s = SavingsAccount("Ali", 10000)
s.deposit(2000)          # inherited from Account, no code repeated
s.add_interest(0.10)     # its own method
print(s.owner, "has Rs", s.balance)

`SavingsAccount` never defines `deposit` or `__init__`, yet both work — Python didn't find them on the child, so it looked *up* to the parent and used those. That upward search is the heart of inheritance.

# 2. Overriding and `super()`

A child can **override** a parent method by defining one with the same name — its version wins. And `super()` lets the child call the parent's version instead of copy-pasting it.

A `CurrentAccount` overrides `withdraw` to allow an overdraft up to a limit:

In [ ]:
class CurrentAccount(Account):
    def __init__(self, owner, balance=0, overdraft=5000):
        super().__init__(owner, balance)     # let Account set owner + balance
        self.overdraft = overdraft           # then add our own attribute

    def withdraw(self, amount):              # override: different rule
        if amount > self.balance + self.overdraft:
            print(f"Refused: past the Rs {self.overdraft} overdraft limit")
            return
        self.balance -= amount

c = CurrentAccount("Zara", 1000, overdraft=3000)
c.withdraw(3500)          # allowed - dips into overdraft
print(c.owner, "balance:", c.balance)

`super().__init__(owner, balance)` runs the parent's constructor, so `CurrentAccount` doesn't repeat the `owner`/`balance` lines. Forget that call and the parent's setup never runs — the attributes it creates simply don't exist:

In [ ]:
# BREAKS ON PURPOSE - __init__ overridden but super() never called
class BrokenAccount(Account):
    def __init__(self, owner):
        self.owner = owner        # forgot to set self.balance

b = BrokenAccount("Hassan")
b.deposit(500)                    # deposit does self.balance += ... but there is no balance

**AttributeError: 'BrokenAccount' object has no attribute 'balance'.** When you override `__init__`, calling `super().__init__(...)` is what keeps the parent's setup intact.

# 3. Polymorphism — same call, different behaviour

**Polymorphism** ("many shapes") means different types respond to the *same* method name in their own way. Your code calls `.speak()` and doesn't care which animal it's holding — each object supplies its own answer.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self):
        return "Woof"

class Cat(Animal):
    def speak(self):
        return "Meow"

# One loop, three types - each speaks for itself
for animal in [Dog("Tommy"), Cat("Kitty"), Animal("Thing")]:
    print(f"{animal.name} says {animal.speak()}")

The loop never checks *what kind* of animal it has. Add a `Cow` class tomorrow with its own `speak()` and this loop works unchanged — that's the flexibility polymorphism buys. The same idea works on the accounts: a mixed list, one loop.

In [ ]:
accounts = [SavingsAccount("Ali", 10000), CurrentAccount("Zara", 1000)]
for acc in accounts:
    acc.withdraw(2000)        # each uses ITS OWN withdraw rule
    print(acc.owner, "->", acc.balance)

# 4. Dunder methods: `__str__` and `__eq__`

**Dunder** ("double underscore") methods let your objects plug into Python's own syntax. You've already met one: `__init__`. Two more you'll use constantly:

- `__str__` — what `print(obj)` shows.
- `__eq__` — what `obj1 == obj2` means.

Without `__str__`, printing an object is useless:

In [ ]:
# WRONG ON PURPOSE - runs, but the output is meaningless
plain = Account("Ali", 5000)
print(plain)         # <__main__.Account object at 0x...>

In [ ]:
class Account:
    def __init__(self, owner, balance=0, number=None):
        self.owner = owner
        self.balance = balance
        self.number = number

    def __str__(self):
        return f"{self.owner}'s account: Rs {self.balance}"

    def __eq__(self, other):
        return self.number == other.number      # same account number = same account

a = Account("Ali", 5000, number="A-001")
b = Account("Ali", 9999, number="A-001")        # different balance, same number
print(a)                # __str__ kicks in
print(a == b)           # __eq__ kicks in: True, same account

Now `print(a)` reads like a receipt and `a == b` asks a real question — same account, even though the balances differ. (`__repr__` is the debugging cousin of `__str__`, shown in lists and the REPL; define it too when you want objects readable everywhere.)

# 5. Real use case: one product family, many shapes

The shop sells physical goods *and* digital licences. Both are products, but they ship differently. Inheritance shares what's common; polymorphism lets the checkout treat them all the same.

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def delivery_note(self):
        return "Delivered"

class PhysicalProduct(Product):
    def delivery_note(self):
        return f"Ship '{self.name}' by courier"

class DigitalProduct(Product):
    def delivery_note(self):
        return f"Email licence key for '{self.name}'"

cart = [PhysicalProduct("Keyboard", 3500), DigitalProduct("Antivirus", 1200)]
for item in cart:
    print(f"Rs {item.price:>5} - {item.delivery_note()}")   # same call, right behaviour each time

# 6. Your turn

**1.** Build a shape family. `Shape` is the base with an `area()` that returns `0`. `Rectangle(width, height)` and `Circle(radius)` each override `area()`. Put a few in a list and print each area in one loop (polymorphism).

**2.** The classic: `Animal` with a `speak()`, then `Dog` and `Cat` overriding it. Loop over a mixed list and print what each says.

**3.** Extend Class 10's account: write `SavingsAccount` and `CurrentAccount` as children of `Account`. `SavingsAccount` overrides nothing but adds `add_interest(rate)`; `CurrentAccount` overrides `withdraw` to allow an overdraft. Use `super().__init__(...)` where you add attributes.

**4.** Give a `Book` class `__str__` (so `print(book)` shows title and author) and `__eq__` (two books are equal when their `isbn` matches, even if other fields differ). Prove it with two `Book` objects that share an ISBN.